In [1]:
%reload_ext autoreload
%autoreload 2 --print

In [2]:
import warnings
warnings.filterwarnings("ignore")

In [3]:
import pandas as pd
from src.paths import TRANSFORMED_DATA_DIR

df= pd.read_parquet(TRANSFORMED_DATA_DIR/"tabular_data.parquet")
df

,rides_previous_672_hour,rides_previous_671_hour,rides_previous_670_hour,rides_previous_669_hour,rides_previous_668_hour,rides_previous_667_hour,rides_previous_666_hour,rides_previous_665_hour,rides_previous_664_hour,rides_previous_663_hour,...,rides_previous_7_hour,rides_previous_6_hour,rides_previous_5_hour,rides_previous_4_hour,rides_previous_3_hour,rides_previous_2_hour,rides_previous_1_hour,pickup_hour,location_id,target_rides_next_hour
0,2.0,3.0,2.0,2.0,0.0,1.0,0.0,0.0,0.0,0.0,...,1.0,1.0,1.0,0.0,0.0,0.0,0.0,2026-01-29,3,0.0
1,0.0,0.0,0.0,0.0,1.0,0.0,2.0,1.0,0.0,0.0,...,1.0,0.0,0.0,0.0,1.0,0.0,0.0,2026-01-30,3,0.0
2,0.0,0.0,0.0,0.0,2.0,0.0,0.0,1.0,0.0,0.0,...,0.0,2.0,2.0,0.0,0.0,1.0,0.0,2026-01-31,3,0.0
3,0.0,0.0,0.0,2.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,1.0,2.0,2026-02-01,3,0.0
4,0.0,0.0,0.0,0.0,1.0,1.0,2.0,2.0,1.0,2.0,...,0.0,3.0,2.0,1.0,0.0,0.0,0.0,2026-02-02,3,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
64185,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,2026-09-26,199,0.0
64186,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,2026-09-27,199,0.0
64187,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,2026-09-28,199,0.0
64188,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,2026-09-29,199,0.0


In [4]:
from datetime import datetime
from src.data_split import train_test_split

X_train, y_train, X_test, y_test = train_test_split(
    df,
    cutoff_date= datetime(2026, 7, 1, 0, 0),
    target_column_name= "target_rides_next_hour"

)

print(f"{X_train.shape=}")
print(f"{X_test.shape=}")
print(f"{y_train.shape=}")
print(f"{y_test.shape=}")

X_train.shape=(40086, 674)
X_test.shape=(24104, 674)
y_train.shape=(40086,)
y_test.shape=(24104,)


In [7]:
import numpy as np
from sklearn.model_selection import KFold, TimeSeriesSplit
from sklearn.pipeline import make_pipeline
from sklearn.metrics import mean_absolute_error
import optuna

from src.model import get_pipeline

def objective(trial: optuna.trial.Trial) -> float:
    """
    Given a set of hyperparameters, it trains a model and computes
    an average validation error based on a TimeSeriesSplit.
    """
    # select hyperparameters
    hyperparams ={
        "metric": "mae",
        "verbose": -1,
        "num_leaves": trial.suggest_int("num_leaves", 2, 256),
        "feature_fraction": trial.suggest_float("feature_fraction:",0.2, 1.0),
        "bagging_fraction": trial.suggest_float("bagging_fraction", 0.2, 1.0),
        "min_child_samples": trial.suggest_int("min_child_samples",3 ,100)
    }

    tss= TimeSeriesSplit(n_splits=4)
    scores= []
    for train_index, val_index in tss.split(X_train):

        # split the data into training and validation sets
        X_train_, X_val_= X_train.iloc[train_index], X_train.iloc[val_index]
        y_train_, y_val_= y_train.iloc[train_index], y_train.iloc[val_index]

        # train the model
        pipeline= get_pipeline(**hyperparams)
        pipeline.fit(X_train_, y_train_)

        # evaluate the model
        y_pred= pipeline.predict(X_val_)
        mae= mean_absolute_error(y_val_, y_pred)

        scores.append(mae)

        # Return the mean score
        return np.array(scores).mean()

In [8]:
# creata an Optuna study and optimize the objective function
study= optuna.create_study(direction="minimize")
study.optimize(objective, n_trials=5)

[I 2026-11-02 13:58:30,463] A new study created in memory with name: no-name-d14ac116-573b-459c-8e75-9c88a7409f48
[I 2026-11-02 13:58:34,366] Trial 0 finished with value: 11.235739376001996 and parameters: {'num_leaves': 2, 'feature_fraction:': 0.8015582920382955, 'bagging_fraction': 0.732871366171022, 'min_child_samples': 95}. Best is trial 0 with value: 11.235739376001996.
[I 2026-11-02 13:58:45,173] Trial 1 finished with value: 10.69942463493792 and parameters: {'num_leaves': 230, 'feature_fraction:': 0.9432488302831776, 'bagging_fraction': 0.8355047763197307, 'min_child_samples': 74}. Best is trial 1 with value: 10.69942463493792.
[I 2026-11-02 13:58:48,034] Trial 2 finished with value: 10.55681936104202 and parameters: {'num_leaves': 22, 'feature_fraction:': 0.4271991193526559, 'bagging_fraction': 0.7298765029079053, 'min_child_samples': 57}. Best is trial 2 with value: 10.55681936104202.
[I 2026-11-02 13:58:56,528] Trial 3 finished with value: 10.61340915458474 and parameters: {'

In [9]:
# extract the best hyperparameters that Optuna found
best_params= study.best_trial.params
print(f"{best_params=}")

best_params={'num_leaves': 22, 'feature_fraction:': 0.4271991193526559, 'bagging_fraction': 0.7298765029079053, 'min_child_samples': 57}


In [10]:
# Retrain the model using the best hyperparmeters found
pipeline= get_pipeline(**best_params)
pipeline.fit(X_train, y_train)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('functiontransformer', ...), ('temporalfeatureengineer', ...), ...]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
Name,Type,Value
"feature_names_in_ feature_names_in_: ndarray of shape (`n_features_in_`,)Names of features seen during :term:`fit`. Only defined if theunderlying estimator exposes such an attribute when fit... versionadded:: 1.0","ndarray[object](674,)","['rides_previous_672_hour','rides_previous_671_hour', 'rides_previous_670_hour',...,'rides_previous_1_hour','pickup_hour', 'location_id']"
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`. Only defined if theunderlying first estimator in `steps` exposes such an attributewhen fit... versionadded:: 0.24,int,674
,"func func: callable, default=NoneThe callable to use for the transformation. This will be passedthe same arguments as transform, with args and kwargs forwarded.If func is None, then func will be the identity function.",<function ave...001C083794540>
,"inverse_func inverse_func: callable, default=NoneThe callable to use for the inverse transformation. This will bepassed the same arguments as inverse transform, with args andkwargs forwarded. If inverse_func is None, then inverse_funcwill be the identity function.",None
,"validate validate: bool, default=FalseIndicate that the input X array should be checked before calling``func``. The possibilities are:- If False, there is no input validation.- If True, then X will be converted to a 2-dimensional NumPy array or sparse matrix. If the conversion is not possible an exception is raised... versionchanged:: 0.22 The default of ``validate`` changed from True to False.",False
,"accept_sparse accept_sparse: bool, default=FalseIndicate that func accepts a sparse matrix as input. If validate isFalse, this has no effect. Otherwise, if accept_sparse is false,sparse matrix inputs will cause an exception to be raised.",False


In [12]:
predictions= pipeline.predict(X_test)

from sklearn.metrics import mean_absolute_error
test_mae= mean_absolute_error(y_test, predictions)
print(f"{test_mae= :.4f}") 

test_mae= 3.6866


In [15]:
# plot a few examples of the predictions vs the actual values
from src.plot import plot_one_sample

plot_one_sample(
    features= X_test,
    targets= y_test,
    example_id= 1500,
    predictions= pd.Series(predictions)
)